# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [2]:
ПРІЗВИЩЕ_ІМЯ = 'Гуменюк Анастасія'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-31'
НОМЕР        = 5       # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [3]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 5   ·   Гуменюк Анастасія   ·   ШІ-31

Ваші вісім стовпців:
   1. Телефон
   2. помер/жив
   3. ЛН
   4. Виписка
   5. spO2 без дот
   6. Прокальцитонін
   7. БРнепр1
   8. Гематокрит1

Ваша пара для аналізу зв’язку: Стать та ступінь КТ

Ваше додаткове питання:
   Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?


In [8]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,Телефон,помер/жив,ЛН,Виписка,spO2 без дот,Прокальцитонін,БРнепр1,Гематокрит1
0,0958469788 дружина Ганна,1,ІІІ-IV,2022-03-10 00:00:00,90.0,0.126,10.2,48.0
1,"0671778486, 0994887663 дружина Оксана",2,ІІ-ІІІ,2022-03-16 00:00:00,83.0,NaN,1.8,42.2
2,"0958768045, чоловік Степан 0951784726",2,ІІ,2022-01-18 00:00:00,78.0,NaN,NaN,40.0
3,0959819331. дочка Валентина 0960845205,2,ІІІ-IV,2022-01-13 00:00:00,82.0,NaN,8.5,22.0
4,"0952294866, дочка Дарина 0678360161",2,ІІ-ІІІ,2022-03-11 00:00:00,80.0,NaN,10.2,40.0
5,0959922983 сестра Софія,1,ІІ-ІІІ,2021-12-26 00:00:00,72.0,NaN,8.5,39.0
6,"0998073702, дружина Наталія 0998382226",2,ІІ-ІІІ,2022-05-11 00:00:00,89.0,NaN,10.2,44.9
7,"0637258771, син Сергій 0968413301",2,І,2022-05-28 00:00:00,90.0,NaN,8.5,38.0


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [9]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Телефон         120 non-null    object 
 1   помер/жив       121 non-null    int64  
 2   ЛН              121 non-null    object 
 3   Виписка         120 non-null    object 
 4   spO2 без дот    120 non-null    float64
 5   Прокальцитонін  46 non-null     float64
 6   БРнепр1         108 non-null    float64
 7   Гематокрит1     115 non-null    float64
dtypes: float64(4), int64(1), object(3)
memory usage: 7.7+ KB


In [10]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- Телефон
    непорожніх 120 з 121 | унікальних 120 | типи: str×97, int×23
    приклади: ["'0958469788 дружина Ганна'", "'0671778486, 0994887663 дружина Оксана'", "'0958768045, чоловік Степан 0951784726'", "'0959819331. дочка Валентина 0960845205'"]

--- помер/жив
    непорожніх 121 з 121 | унікальних 2 | типи: int×121
    приклади: ['np.int64(1)', 'np.int64(2)']

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- spO2 без дот
    непорожніх 120 з 121 | унікальних 33 | типи: float×120
    приклади: ['np.float64(90.0)', 'np.float64(83.0)', 'np.float64(78.0)', 'np.float64(82.0)']

--- Прокальцитонін
    непорожніх 46 з 121 | унікальних 17 | типи: float×46
    прик

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [11]:
ПАСПОРТ = {
    'Телефон': (
        'ідентифікатор',
        'Номер телефону використовується для ідентифікації людини, а не для вимірювання величини.'
    ),

    'помер/жив': (
        'номінальна',
        'Значення 1 і 2 позначають категорії результату лікування, між якими немає кількісної відстані.'
    ),

    'ЛН': (
        'порядкова',
        'Ступені мають природний порядок від меншого до більшого, але відстані між ними не визначені.'
    ),

    'Виписка': (
        'інтервальна',
        'Між датами можна змістовно визначати різницю в днях, але природного нуля календарна дата не має.'
    ),

    'spO2 без дот': (
        'відношень',
        'Насичення киснем вимірюється у відсотках і має природний нуль.'
    ),

    'Прокальцитонін': (
        'відношень',
        'Концентрація має природний нуль, а відношення між числовими значеннями мають зміст.'
    ),

    'БРнепр1': (
        'відношень',
        'Числовий показник має природний нуль, тому можна змістовно порівнювати не лише різниці, а й відношення.'
    ),

    'Гематокрит1': (
        'відношень',
        'Гематокрит є кількісним показником із природним нулем, тому відношення його значень мають зміст.'
    )
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування'])
паспорт_df

,Стовпець,Шкала,Обґрунтування
0,Телефон,ідентифікатор,Номер телефону використовується для ідентифікації людини...
1,помер/жив,номінальна,Значення 1 і 2 позначають категорії результату лікування...
2,ЛН,порядкова,"Ступені мають природний порядок від меншого до більшого,..."
3,Виписка,інтервальна,"Між датами можна змістовно визначати різницю в днях, але..."
4,spO2 без дот,відношень,Насичення киснем вимірюється у відсотках і має природний...
5,Прокальцитонін,відношень,"Концентрація має природний нуль, а відношення між числов..."
6,БРнепр1,відношень,"Числовий показник має природний нуль, тому можна змістов..."
7,Гематокрит1,відношень,"Гематокрит є кількісним показником із природним нулем, т..."


In [12]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [15]:
# Порядок категорій для ЛН
порядок_ЛН = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV']


def допустима_статистика(стовпець, шкала):
    з = df[стовпець].dropna()

    частоти = з.value_counts()

    # Якщо всі значення зустрічаються лише один раз,
    # змістовної моди немає
    if len(частоти) > 0 and частоти.max() == 1:
        мода = 'відсутня'
    else:
        мода = з.mode().tolist()

    р = {
        'непорожніх': len(з),
        'унікальних': з.nunique(),
        'мода': мода,
        'частоти': частоти.to_dict()
    }

    ш = шкала.strip().lower()

    # Ідентифікатор і номінальна — тільки мода та частоти
    if ш in ('ідентифікатор', 'номінальна'):
        return р

    # Порядкова — + медіана і квантилі
    if ш == 'порядкова':
        if стовпець == 'ЛН':
            cat = pd.Categorical(
                з,
                categories=порядок_ЛН,
                ordered=True
            )

            коди = pd.Series(cat.codes[cat.codes >= 0])

            q = коди.quantile([0.25, 0.5, 0.75])

            р['медіана'] = порядок_ЛН[round(q.loc[0.5])]
            р['Q1'] = порядок_ЛН[round(q.loc[0.25])]
            р['Q3'] = порядок_ЛН[round(q.loc[0.75])]

        return р

    # Інтервальна
    if ш == 'інтервальна':
        дати = pd.to_datetime(з, errors='coerce').dropna()

        частоти_дат = дати.value_counts()

        if частоти_дат.max() == 1:
            р['мода'] = 'відсутня'
        else:
            р['мода'] = [d.strftime('%Y-%m-%d') for d in дати.mode()]

        р['середнє'] = дати.mean()
        р['стандартне відхилення'] = pd.Series(дати).std()

        return р

    # Шкала відношень
    if ш == 'відношень':
        x = pd.to_numeric(
            з,
            errors='coerce'
        ).dropna()

        р['медіана'] = x.median()
        р['Q1'] = x.quantile(0.25)
        р['Q3'] = x.quantile(0.75)
        р['середнє'] = x.mean()
        р['стандартне відхилення'] = x.std()

        if x.mean() != 0:
            р['коефіцієнт варіації, %'] = x.std() / x.mean() * 100
        else:
            р['коефіцієнт варіації, %'] = None

        return р

    return р


for c in МОЇ:
    print(f'--- {c} ({ПАСПОРТ[c][0]})')
    print(допустима_статистика(c, ПАСПОРТ[c][0]))
    print()

--- Телефон (ідентифікатор)
{'непорожніх': 120, 'унікальних': 120, 'мода': 'відсутня', 'частоти': {'0958469788 дружина Ганна': 1, '0671778486, 0994887663 дружина Оксана': 1, '0958768045, чоловік Степан 0951784726': 1, '0959819331. дочка Валентина 0960845205': 1, '0952294866, дочка Дарина 0678360161': 1, '0959922983 сестра Софія': 1, '0998073702, дружина Наталія 0998382226': 1, '0637258771, син Сергій 0968413301': 1, '0963194893, сестра Катерина 0964948214': 1, '0635244106 син Микола': 1, 504118955: 1, 955939761: 1, 635060454: 1, '0679110285 подруга Зоряна': 1, '0993228088, син Степан 0954230048': 1, 961280845: 1, 990370382: 1, 'син владислав 0997879432': 1, 'син Ярослав 0638421083': 1, '0638440932, дружина Ганна 0634845263': 1, 688280258: 1, 'син Володимир 0679932157': 1, '0675795111, дочка 0954432568': 1, '0966327926, син 0635154742': 1, 'дружина Юлія 0950856517': 1, '0675996775, син Сергій 0500314503': 1, '0953924196, чол.Володимир 0998982104': 1, '0954754072, дочка Христина 05075473

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [16]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Телефон,120,120,0958469788 дружина Ганна,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
помер/жив,121.0,NaN,NaN,NaN,1.495868,0.502062,1.0,1.0,1.0,2.0,2.0
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
spO2 без дот,120.0,NaN,NaN,NaN,82.049167,12.208249,0.9,78.75,84.0,89.0,98.0
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,3.010375,0.043,0.1,0.1,0.1205,18.22
БРнепр1,108.0,NaN,NaN,NaN,10.24537,2.398924,1.8,8.5,10.2,11.35,22.1
Гематокрит1,115.0,NaN,NaN,NaN,39.092174,7.324773,10.0,36.0,40.0,43.5,56.0


In [17]:
БЕЗГЛУЗДІ = [
    (
        'помер/жив',
        'середнє',
        df['помер/жив'].mean(),
        'Значення 1 і 2 є лише кодами категорій, тому середнє арифметичне між ними не має змісту.'
    ),

    (
        'помер/жив',
        'стандартне відхилення',
        df['помер/жив'].std(),
        'Стандартне відхилення спирається на числові відстані між значеннями, а для номінальних категорій такої відстані немає.'
    ),

    (
        'помер/жив',
        'медіана',
        df['помер/жив'].median(),
        'Для номінальної змінної категорії не мають природного порядку, тому медіана не має змісту.'
    ),

    (
        'помер/жив',
        'Q1',
        df['помер/жив'].quantile(0.25),
        'Перший квартиль потребує впорядкованих значень, а категорії результату лікування не мають природного порядку.'
    ),

    (
        'помер/жив',
        'Q3',
        df['помер/жив'].quantile(0.75),
        'Третій квартиль також не має змісту для номінальної змінної, бо її категорії не можна змістовно розташувати за величиною.'
    )
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'

pd.DataFrame(
    БЕЗГЛУЗДІ,
    columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде']
)

,Стовпець,Статистика,Значення,Чому безглузде
0,помер/жив,середнє,1.495868,"Значення 1 і 2 є лише кодами категорій, тому середнє ари..."
1,помер/жив,стандартне відхилення,0.502062,Стандартне відхилення спирається на числові відстані між...
2,помер/жив,медіана,1.000000,Для номінальної змінної категорії не мають природного по...
3,помер/жив,Q1,1.000000,"Перший квартиль потребує впорядкованих значень, а катего..."
4,помер/жив,Q3,2.000000,Третій квартиль також не має змісту для номінальної змін...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [18]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: Стать × ступінь КТ

Стать: шкала — не у вашому списку
ступінь КТ: шкала — не у вашому списку

  Стать ступінь КТ
0     ч    сердній
1     ч   середній
2     ж  критичний
3     ж  критичний
4     ж        NaN
5     ж   середній


In [21]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, pearsonr

# Беремо потрібні два стовпці
пара = df[['Стать', 'ступінь КТ']].copy()

# Уніфікуємо стать
пара['Стать'] = (
    пара['Стать']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({
        'ч': 'чоловіча',
        'чол': 'чоловіча',
        'ж': 'жіноча',
        'жін': 'жіноча'
    })
)

# Уніфікуємо ступінь КТ
пара['ступінь КТ'] = (
    пара['ступінь КТ']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({
        'сердній': 'середній',
        'серед ступ': 'середній',
        'серед ступеня': 'середній',
        'легка': 'легкий'
    })
)

# Видаляємо пропущені значення
пара = пара.dropna()

# Таблиця спряженості
таблиця = pd.crosstab(пара['Стать'], пара['ступінь КТ'])
display(таблиця)

# χ²
chi2, p, dof, expected = chi2_contingency(таблиця)

# Коефіцієнт Крамера V
n = таблиця.to_numpy().sum()
r, k = таблиця.shape

V = np.sqrt(chi2 / (n * min(r - 1, k - 1)))

ОБРАНА_МІРА = 'Коефіцієнт Крамера V'

ЧОМУ_САМЕ_ВОНА = (
    'Стать є номінальною змінною, а ступінь КТ - порядковою. '
    'Для такої пари шкал використовують коефіцієнт Крамера V, '
    'який оцінює силу зв’язку між категоріальними змінними.'
)

print('Обрана міра:', ОБРАНА_МІРА)
print('Cramer V =', round(V, 3))
print('chi² =', round(chi2, 3))
print('p-value =', round(p, 4))
print(ЧОМУ_САМЕ_ВОНА)

ступінь КТ,важкий,вкрай важкий,критичний,легкий,середній
Стать,,,,,
жіноча,4,2,3,4,8
чоловіча,6,2,0,3,10


Обрана міра: Коефіцієнт Крамера V
Cramer V = 0.299
chi² = 3.765
p-value = 0.4387
Стать є номінальною змінною, а ступінь КТ - порядковою. Для такої пари шкал використовують коефіцієнт Крамера V, який оцінює силу зв’язку між категоріальними змінними.


In [20]:
# Тепер — контрприклад: застосуйте кореляцію Пірсона незалежно від шкал
# і поясніть, чому отримане число не можна використовувати.


стать_код = пара['Стать'].map({
    'жіноча': 0,
    'чоловіча': 1
})

кт_код = пара['ступінь КТ'].map({
    'легкий': 1,
    'середній': 2,
    'важкий': 3,
    'критичний': 4,
    'вкрай важкий': 5
})

r, p_pearson = pearsonr(стать_код, кт_код)

print('Кореляція Пірсона:', round(r, 3))
print('p-value:', round(p_pearson, 4))

Кореляція Пірсона: -0.063
p-value: 0.6932


Кореляція Пірсона для цієї пари недопустима, оскільки стать є номінальною змінною, а ступінь КТ - порядковою. Числове кодування категорій не створює справжньої кількісної шкали, тому отриманий коефіцієнт Пірсона не має коректної інтерпретації. Для оцінки зв’язку використано коефіцієнт Крамера V.

---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [22]:
ПІДГОТОВКА = {
    'Телефон': (
        'не використовувати в моделі, оскільки це ідентифікатор'
    ),

    'помер/жив': (
        'one-hot кодування, оскільки це номінальна змінна без природного порядку'
    ),

    'ЛН': (
        'порядкове кодування зі збереженням природного порядку категорій'
    ),

    'Виписка': (
        'перетворення дати у тривалість від початкової дати'
    ),

    'spO2 без дот': (
        'стандартизація, оскільки це кількісна змінна шкали відношень'
    ),

    'Прокальцитонін': (
        'стандартизація; за потреби можливе логарифмування через великий розкид значень'
    ),

    'БРнепр1': (
        'стандартизація кількісної змінної'
    ),

    'Гематокрит1': (
        'стандартизація кількісної змінної'
    )
}

for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Як готуємо']
)

,Стовпець,Шкала,Як готуємо
0,Телефон,ідентифікатор,"не використовувати в моделі, оскільки це ідентифікатор"
1,помер/жив,номінальна,"one-hot кодування, оскільки це номінальна змінна без при..."
2,ЛН,порядкова,порядкове кодування зі збереженням природного порядку ка...
3,Виписка,інтервальна,перетворення дати у тривалість від початкової дати
4,spO2 без дот,відношень,"стандартизація, оскільки це кількісна змінна шкали відно..."
5,Прокальцитонін,відношень,стандартизація; за потреби можливе логарифмування через ...
6,БРнепр1,відношень,стандартизація кількісної змінної
7,Гематокрит1,відношень,стандартизація кількісної змінної


In [23]:
# ваш код: реалізуйте підготовку для трьох стовпців різних шкал
from sklearn.preprocessing import StandardScaler

df_model = df.copy()


# 1. НОМІНАЛЬНА ШКАЛА — помер/жив
# One-hot кодування
помер_жив_encoded = pd.get_dummies(
    df_model['помер/жив'],
    prefix='результат',
    dtype=int
)

df_model = pd.concat([df_model, помер_жив_encoded], axis=1)


# 2. ПОРЯДКОВА ШКАЛА — ЛН
# Задаємо природний порядок категорій
порядок_ЛН = {
    'І': 1,
    'І-ІІ': 2,
    'ІІ': 3,
    'ІІ-ІІІ': 4,
    'ІІІ': 5,
    'ІІІ-IV': 6
}

df_model['ЛН_код'] = df_model['ЛН'].map(порядок_ЛН)


# 3. ІНТЕРВАЛЬНА ШКАЛА — Виписка
# Перетворюємо дату на кількість днів від найранішої дати
дати = pd.to_datetime(
    df_model['Виписка'],
    errors='coerce'
)

початкова_дата = дати.min()

df_model['днів_від_початку'] = (
    дати - початкова_дата
).dt.days


# 4. ШКАЛА ВІДНОШЕНЬ — spO2 без дот
# Перевіряємо варіативність перед стандартизацією
x = pd.to_numeric(
    df_model['spO2 без дот'],
    errors='coerce'
)

if x.dropna().nunique() <= 1:
    print('УВАГА: spO2 без дот — сталий стовпець, стандартизація неможлива.')
else:
    scaler = StandardScaler()

    mask = x.notna()

    df_model.loc[mask, 'spO2_стандартизований'] = scaler.fit_transform(
        x[mask].to_numpy().reshape(-1, 1)
    ).ravel()


# Показуємо результат
df_model[
    [
        'помер/жив',
        'результат_1',
        'результат_2',
        'ЛН',
        'ЛН_код',
        'Виписка',
        'днів_від_початку',
        'spO2 без дот',
        'spO2_стандартизований'
    ]
].head(10)

,помер/жив,результат_1,результат_2,ЛН,ЛН_код,Виписка,днів_від_початку,spO2 без дот,spO2_стандартизований
0,1,1,0,ІІІ-IV,6,2022-03-10 00:00:00,317.0,90.0,0.653998
1,2,0,1,ІІ-ІІІ,4,2022-03-16 00:00:00,323.0,83.0,0.078211
2,2,0,1,ІІ,3,2022-01-18 00:00:00,266.0,78.0,-0.333065
3,2,0,1,ІІІ-IV,6,2022-01-13 00:00:00,261.0,82.0,-0.004044
4,2,0,1,ІІ-ІІІ,4,2022-03-11 00:00:00,318.0,80.0,-0.168555
5,1,1,0,ІІ-ІІІ,4,2021-12-26 00:00:00,243.0,72.0,-0.826597
6,2,0,1,ІІ-ІІІ,4,2022-05-11 00:00:00,379.0,89.0,0.571743
7,2,0,1,І,1,2022-05-28 00:00:00,396.0,90.0,0.653998
8,2,0,1,ІІ,3,2022-05-13 00:00:00,381.0,78.0,-0.333065
9,1,1,0,ІІІ,5,2021-11-19 00:00:00,206.0,80.0,-0.168555


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [24]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [25]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [26]:
ВИПРАВЛЕННЯ = [
    (
        1,
        'Для змінної "помер/жив" обчислено середнє, хоча це номінальна шкала.',
        'Числа 1 і 2 є лише кодами категорій "помер" і "вижив", тому значення 1.50 не означає проміжний результат лікування.',
        'Потрібно використовувати частоти або частки категорій: 61 померлий і 60 живих.'
    ),

    (
        2,
        'Значення статі не очищено перед підрахунком категорій.',
        'Записи "ж" і "жін" означають одну стать, а "ч" і "чол" — іншу, тому 4 унікальні значення не означають 4 статі.',
        'Спочатку об’єднати синонімічні записи. Після очищення: 75 жінок і 46 чоловіків.'
    ),

    (
        3,
        'Для порядкової змінної "ступінь КТ" категорії впорядковано алфавітно і обчислено середнє.',
        'Алфавітний порядок не відповідає тяжкості, а відстані між ступенями не є однаковими, тому середнє кодів не має коректного змісту.',
        'Задати природний порядок: легкий < середній < важкий < критичний < вкрай важкий і використовувати медіану або моду.'
    ),

    (
        4,
        'Стовпець "КТ" напряму перетворено через pd.to_numeric, хоча значення записані у різних форматах.',
        'Записи типу "20-30%" або "70-80%" перетворюються на NaN, тому середнє рахується лише за частиною даних і не описує всю вибірку.',
        'Спочатку привести всі значення КТ до єдиного числового формату, наприклад використати середину інтервалу для записів 20-30%.'
    ),

    (
        5,
        'Середню сатурацію обчислено без перевірки якості значень.',
        'У стовпці є значення 0.9, яке різко відрізняється від решти значень 50-98 і, ймовірно, записане в іншому масштабі.',
        'Спочатку перевірити та виправити підозріле значення; якщо 0.9 означає 90%, після виправлення середня сатурація становить близько 82.79%.'
    ),

    (
        6,
        'Для календарних дат обчислено коефіцієнт варіації.',
        'Дата має інтервальну, а не шкалу відношень: нульова точка календаря умовна, тому відношення стандартного відхилення до середньої дати не має змісту.',
        'Для дат можна аналізувати розкид у днях, діапазон або стандартне відхилення тривалостей від обраної початкової дати, але не CV абсолютних дат.'
    ),

    (
        7,
        'Між статтю та результатом лікування застосовано кореляцію Пірсона.',
        'Обидві змінні є номінальними, а числові коди 0 і 1 не створюють кількісної шкали, тому коефіцієнт Пірсона тут не є правильною мірою зв’язку.',
        'Потрібно використати таблицю спряженості, критерій χ² і коефіцієнт Крамера V або φ для двох бінарних змінних.'
    ),

    (
        8,
        'Нормалізовано всі числові стовпці без урахування їх змісту та варіативності.',
        'Серед числових стовпців можуть бути коди категорій та сталі ознаки; для сталого стовпця max-min = 0 і нормалізація дає NaN.',
        'Нормалізувати лише справді кількісні ознаки та перед цим перевіряти, що max != min.'
    ),

    (
        9,
        'Для очищення використано df.dropna() для всіх 77 ознак одночасно.',
        'Через велику кількість пропусків після такого видалення залишається 0 рядків, тому набір не є придатним для моделі.',
        'Працювати з потрібними для моделі ознаками та обробляти пропуски вибірково: видаляти або імпутувати їх залежно від змінної.'
    ),

    (
        10,
        'Дата розтину використана як предиктор летальності.',
        'Дата розтину з’являється вже після смерті пацієнта, тому вона фактично містить інформацію про цільову змінну і створює витік даних.',
        'Виключити "Дата розтину" та інші післярезультатні ознаки з моделі і використовувати лише дані, доступні до настання результату лікування.'
    )
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'

for p in ВИПРАВЛЕННЯ:
    assert len(p) == 4 and all(str(x).strip() for x in p), 'Заповніть усі чотири поля'
    assert len(str(p[2])) >= 30 and len(str(p[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(
    ВИПРАВЛЕННЯ,
    columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно']
)

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"Для змінної ""помер/жив"" обчислено середнє, хоча це номін...","Числа 1 і 2 є лише кодами категорій ""помер"" і ""вижив"", т...",Потрібно використовувати частоти або частки категорій: 6...
1,2,Значення статі не очищено перед підрахунком категорій.,"Записи ""ж"" і ""жін"" означають одну стать, а ""ч"" і ""чол"" —...",Спочатку об’єднати синонімічні записи. Після очищення: 7...
2,3,"Для порядкової змінної ""ступінь КТ"" категорії впорядкова...","Алфавітний порядок не відповідає тяжкості, а відстані мі...",Задати природний порядок: легкий < середній < важкий < к...
3,4,"Стовпець ""КТ"" напряму перетворено через pd.to_numeric, х...","Записи типу ""20-30%"" або ""70-80%"" перетворюються на NaN,...",Спочатку привести всі значення КТ до єдиного числового ф...
4,5,Середню сатурацію обчислено без перевірки якості значень.,"У стовпці є значення 0.9, яке різко відрізняється від ре...",Спочатку перевірити та виправити підозріле значення; якщ...
5,6,Для календарних дат обчислено коефіцієнт варіації.,"Дата має інтервальну, а не шкалу відношень: нульова точк...","Для дат можна аналізувати розкид у днях, діапазон або ст..."
6,7,Між статтю та результатом лікування застосовано кореляці...,"Обидві змінні є номінальними, а числові коди 0 і 1 не ст...","Потрібно використати таблицю спряженості, критерій χ² і ..."
7,8,Нормалізовано всі числові стовпці без урахування їх зміс...,Серед числових стовпців можуть бути коди категорій та ст...,Нормалізувати лише справді кількісні ознаки та перед цим...
8,9,Для очищення використано df.dropna() для всіх 77 ознак о...,Через велику кількість пропусків після такого видалення ...,Працювати з потрібними для моделі ознаками та обробляти ...
9,10,Дата розтину використана як предиктор летальності.,"Дата розтину з’являється вже після смерті пацієнта, тому...","Виключити ""Дата розтину"" та інші післярезультатні ознаки..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [27]:
# ваш код: три правильні обчислення замість хибних
# 1. Розділ 1 — результат лікування:
# замість середнього показуємо частоти і частки

результат = df['помер/жив'].value_counts().sort_index()
частки = df['помер/жив'].value_counts(normalize=True).sort_index() * 100

print('Результат лікування:')
print('Померли:', результат.get(1, 0),
      f'({частки.get(1, 0):.1f}%)')
print('Вижили:', результат.get(2, 0),
      f'({частки.get(2, 0):.1f}%)')

print()


# 2. Розділ 2 — очищаємо стать

стать_чиста = (
    df['Стать']
    .astype('string')
    .str.strip()
    .str.lower()
    .replace({
        'ж': 'жіноча',
        'жін': 'жіноча',
        'ч': 'чоловіча',
        'чол': 'чоловіча'
    })
)

print('Стать після очищення:')
print(стать_чиста.value_counts())

print()


# 3. Розділ 5 — перевіряємо сатурацію

сатурація = df['spO2 без дот'].copy()

print('До виправлення:')
print('мінімум =', сатурація.min())
print('середнє =', round(сатурація.mean(), 2))

# 0.9, імовірно, записано як частку замість 90%
сатурація = сатурація.replace(0.9, 90)

print('\nПісля виправлення:')
print('мінімум =', сатурація.min())
print('середнє =', round(сатурація.mean(), 2))
print('медіана =', round(сатурація.median(), 2))

Результат лікування:
Померли: 61 (50.4%)
Вижили: 60 (49.6%)

Стать після очищення:
Стать
жіноча      75
чоловіча    46
Name: count, dtype: Int64

До виправлення:
мінімум = 0.9
середнє = 82.05

Після виправлення:
мінімум = 50.0
середнє = 82.79
медіана = 84.0


---
# Етап 7. Висновки та індивідуальне питання

In [28]:
ВИСНОВКИ = [
    (
        'У вибірці результат лікування майже рівномірно розподілений між двома категоріями: '
        'значення 1 зустрічається 61 раз, а значення 2 — 60 разів.'
    ),

    (
        'Для легеневої недостатності найчастіше зустрічається ступінь ІІ-ІІІ — 59 випадків; '
        'медіанний ступінь також дорівнює ІІ-ІІІ, а центральні 50% значень лежать між ІІ та ІІ-ІІІ.'
    ),

    (
        'Середня сатурація без додаткового кисню становить приблизно 82.05%, медіана — 84%, '
        'але в даних є підозріле значення 0.9, яке може бути помилкою введення та впливати на статистики.'
    ),

    (
        'Прокальцитонін має дуже нерівномірний розподіл: медіана становить 0.1, тоді як середнє близько 0.75, '
        'а коефіцієнт варіації майже 400%, що пов’язано з кількома дуже великими значеннями.'
    )
]

assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'

for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У вибірці результат лікування майже рівномірно розподілений між двома категоріями: значення 1 зустрічається 61 раз, а значення 2 — 60 разів.
2. Для легеневої недостатності найчастіше зустрічається ступінь ІІ-ІІІ — 59 випадків; медіанний ступінь також дорівнює ІІ-ІІІ, а центральні 50% значень лежать між ІІ та ІІ-ІІІ.
3. Середня сатурація без додаткового кисню становить приблизно 82.05%, медіана — 84%, але в даних є підозріле значення 0.9, яке може бути помилкою введення та впливати на статистики.
4. Прокальцитонін має дуже нерівномірний розподіл: медіана становить 0.1, тоді як середнє близько 0.75, а коефіцієнт варіації майже 400%, що пов’язано з кількома дуже великими значеннями.


In [31]:
print('ВАШЕ ПИТАННЯ:'); print(В['питання']); print()
ВІДПОВІДЬ = """
Коефіцієнт кореляції Пірсона не можна коректно використовувати між статтю та результатом лікування,
оскільки обидві змінні є номінальними. Числа, якими можуть бути закодовані категорії, наприклад
0 і 1 або 1 і 2, є лише умовними позначеннями та не утворюють кількісної шкали з осмисленими
відстанями між значеннями. Тому отриманий коефіцієнт Пірсона залежав би від способу кодування
і не мав би коректної інтерпретації. Для двох категоріальних змінних потрібно побудувати таблицю
спряженості, застосувати критерій хі-квадрат та оцінити силу зв'язку коефіцієнтом Крамера V.
Якщо обидві змінні мають лише по дві категорії, також можна використовувати коефіцієнт фі.
"""
assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'
print(ВІДПОВІДЬ)

ВАШЕ ПИТАННЯ:
Чому не можна обчислити коефіцієнт кореляції Пірсона між статтю та результатом лікування? Яку міру зв’язку слід узяти замість нього?


Коефіцієнт кореляції Пірсона не можна коректно використовувати між статтю та результатом лікування,
оскільки обидві змінні є номінальними. Числа, якими можуть бути закодовані категорії, наприклад
0 і 1 або 1 і 2, є лише умовними позначеннями та не утворюють кількісної шкали з осмисленими
відстанями між значеннями. Тому отриманий коефіцієнт Пірсона залежав би від способу кодування
і не мав би коректної інтерпретації. Для двох категоріальних змінних потрібно побудувати таблицю
спряженості, застосувати критерій хі-квадрат та оцінити силу зв'язку коефіцієнтом Крамера V.
Якщо обидві змінні мають лише по дві категорії, також можна використовувати коефіцієнт фі.



---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [33]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних':        'ні',
    'етап 2 — допустимі статистики':   'код, перевірено',
    'етап 3 — безглузді числа':        'ні',
    'етап 4 — міра зв’язку':           'код, перевірено',
    'етап 5 — підготовка до моделі':   'код, перевірено',
    'етап 6 — виправлення аналізу':    'код, перевірено',
    'етап 7 — висновки й відповідь':   'текст, перероблено',
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = '''
Перевірила правильність визначення шкал змінних, допустимість обчислених статистик,
вибір міри зв’язку для заданої пари змінних, коректність підготовки ознак до моделі,
а також правильність знайдених і виправлених помилок у чужому аналізі.
Окремо перевірила інтерпретацію отриманих статистичних показників і висновків.
'''
ПІДТВЕРДЖУЮ = True     # я можу пояснити кожен рядок свого коду
                        # і внести в нього зміни на вимогу під час захисту

In [34]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 5 готова до захисту.
Гуменюк Анастасія, ШІ-31


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.